In [1]:
import sys
sys.path.append(r'../')
import numpy as np
import matplotlib.pyplot as plt


from src.d_CSL import GcStar
from src.d_CSL import Visualize_on_topography
from src.dataload import *
from src.dataPreProcessing import replace_nan

# if variables are changed on the fly 
from reloading import reloading
from tqdm.notebook import tqdm

## Analysis

In [ ]:
# %%time
metrics, N_PASTS, num = [], list(range(1, 8)), 20
INFs = {}
for a in tqdm(range(num)):
    n_neur = np.random.randint(25, 35, 1)[0] 
    l = np.random.randint(5000, 15000, 1)[0]
    alpha, beta, n_lags = 0.01, 0.001, 1
    
    
    # simulate data
    
    A = adj_mtx(n_neur)
    noise = continuous_noise_fun(num = n_neur, l = l)
    X = np.zeros((A.shape[0], l)).T
    X[0] = np.random.randn(A.shape[0])
    for i, row in enumerate(X[:-1]):
        X[i+1] = A @ X[i] + np.random.normal(0, 0.25, A.shape[0]) + noise[:, i]
    X=X.T
    
    print(f'Data {a} of {num} with {X.shape[0]} vars')
    
    # fit algorithm to data
    met = np.zeros((len(N_PASTS),4))
    Infs = []
    for k, n in enumerate(N_PASTS):
        print(n)
        
        # intantiate the GcStar()
        gcstar = GcStar(n_perm = 1000, n_pasts = n, 
                        n_lags = n_lags, temporal = True, method = ' ')
        
        gcstar.fit(X, verbose=1)
        
        # compute metrics
        Infs.append(gcstar.get_connectivity_matrix(simulation=True))
        gcstar.compute_confusion_matrix(A, simulation = True)
        met[k] = gcstar.compute_metrics()
    metrics.append(met)
    INFs[a] = Infs

  0%|          | 0/20 [00:00<?, ?it/s]

INFO:src.d_CSL:Starting correlation_func
INFO:src.d_CSL:Starting inv_correlation_func
INFO:src.d_CSL:Waiting for correlation_func and inv_correlation_func to complete


Data 0 of 20 with 27 vars
1


C:\Users\sadiq\Documents\GC-extension\examples\..\src\d_CSL.py:32: NumbaExperimentalFeatureWarning: Use of isinstance() detected. This is an experimental feature.
  corr_1 = np.corrcoef(x, y)[1, 0]
INFO:src.d_CSL:Step 1/1458 (0.07% complete)
INFO:src.d_CSL:Step 2/1458 (0.14% complete)
INFO:src.d_CSL:Step 3/1458 (0.21% complete)
INFO:src.d_CSL:Step 4/1458 (0.27% complete)
INFO:src.d_CSL:Step 5/1458 (0.34% complete)
INFO:src.d_CSL:Step 6/1458 (0.41% complete)
INFO:src.d_CSL:Step 7/1458 (0.48% complete)
INFO:src.d_CSL:Step 8/1458 (0.55% complete)
INFO:src.d_CSL:Step 9/1458 (0.62% complete)
INFO:src.d_CSL:Step 10/1458 (0.69% complete)
INFO:src.d_CSL:Step 11/1458 (0.75% complete)
INFO:src.d_CSL:Step 12/1458 (0.82% complete)
INFO:src.d_CSL:Step 13/1458 (0.89% complete)
INFO:src.d_CSL:Step 14/1458 (0.96% complete)
INFO:src.d_CSL:Step 15/1458 (1.03% complete)
INFO:src.d_CSL:Step 16/1458 (1.10% complete)
INFO:src.d_CSL:Step 17/1458 (1.17% complete)
INFO:src.d_CSL:Step 18/1458 (1.23% complete)
I

In [ ]:
for met in metrics:
    print(met)

In [ ]:
Metrics = np.zeros((num,7,4))
for i, met in enumerate(metrics):
    Metrics[i] = met
np.mean(Metrics[:,0,0])

In [ ]:
np.mean(Metrics,axis=0)

In [ ]:
fig, ax = plt.subplots(1, 4, figsize = (12, 3))
labels = ['Accuracy', 'Precision', 'Recall', 'FPR']
lab = ['$\mu_{acc}$','$\mu_{prec}$', '$\mu_{rec}$', '$\mu_{FPR}$']
for i in range(Metrics.shape[2]):
    ax[i].plot(range(1, 1 + Metrics.shape[1]), 
               np.mean(Metrics, axis = 0)[:, i],
               marker = '.', label = lab[i])
    
    ax[i].errorbar(range(1, 1 + Metrics.shape[1]), 
                   np.mean(Metrics, axis = 0)[:, i], 
                   yerr = np.std(Metrics, axis = 0)[:, i], 
                   fmt='o')
    ax[i].set_xticks(range(1,1+Metrics.shape[1]), 
                     list(range(1, 1+Metrics.shape[1])))
    ax[i].set_xlabel('$n_{pasts}$')
    ax[i].set_ylabel('%', rotation = 180)
    ax[i].set_title(f'{labels[i]}')
    ax[i].legend()
    plt.tight_layout()